# Ten-year CHD model comparison

This workflow tunes Logistic Regression, Random Forest, and XGBoost using stratified five-fold cross-validation. The held-out test set is used only for the final comparison.


In [2]:
import json
from pathlib import Path

import joblib
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

RANDOM_STATE = 42
DATA_PATH = Path(r"C:\Users\rohan\Documents\Early_cvd_dectector\data\framingham.csv")
MODEL_PATH = Path(r"C:\Users\rohan\Documents\Early_cvd_dectector\models\best_model.joblib")
METADATA_PATH = Path(r"C:\Users\rohan\Documents\Early_cvd_dectector\models\best_model_metadata.json")


In [3]:
df = pd.read_csv(DATA_PATH)
X = df.drop(columns="TenYearCHD")
y = df["TenYearCHD"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(f"Training rows: {len(X_train):,}; test rows: {len(X_test):,}")
print(f"CHD prevalence: {y.mean():.1%}")


Training rows: 3,392; test rows: 848
CHD prevalence: 15.2%


In [6]:
# Pipelines keep imputation and any required scaling inside cross-validation.
# Tree models do not need scaling; scaling is used only for Logistic Regression.
pipelines = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)),
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("classifier", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=1, class_weight="balanced")),
    ]),
    "XGBoost": Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("classifier", XGBClassifier(
            random_state=RANDOM_STATE, n_jobs=1, eval_metric="logloss",
            objective="binary:logistic", tree_method="hist",
        )),
    ]),
}

param_grids = {
    "Logistic Regression": {"classifier__C": [0.01, 0.1, 1.0, 10.0]},
    "Random Forest": {
        "classifier__n_estimators": [50, 100],
        "classifier__max_depth": [None, 8, 16],
    },
    "XGBoost": {
        "classifier__n_estimators": [50, 100],
        "classifier__max_depth": [3, 5],
        "classifier__learning_rate": [0.03, 0.1],
    },
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


In [8]:
# Tune, evaluate, and retain every fitted GridSearchCV object.
results = {}
for name, pipeline in pipelines.items():
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grids[name],
        scoring="accuracy",
        cv=cv,
        n_jobs=1,
        refit=True,
        return_train_score=False,
    )
    search.fit(X_train, y_train)

    test_predictions = search.predict(X_test)
    test_probabilities = search.predict_proba(X_test)[:, 1]
    results[name] = {
        "search": search,
        "cv_score": search.best_score_,
        "test_accuracy": accuracy_score(y_test, test_predictions),
        "test_roc_auc": roc_auc_score(y_test, test_probabilities),
        "best_params": search.best_params_,
    }

    print(f"\n{name}")
    print(f"  Best 5-fold CV accuracy: {search.best_score_:.3f}")
    print(f"  Test accuracy:           {results[name]['test_accuracy']:.3f}")
    print(f"  Test ROC-AUC:            {results[name]['test_roc_auc']:.3f}")
    print(f"  Best hyperparameters:    {search.best_params_}")



Logistic Regression
  Best 5-fold CV accuracy: 0.856
  Test accuracy:           0.843
  Test ROC-AUC:            0.694
  Best hyperparameters:    {'classifier__C': 1.0}

Random Forest
  Best 5-fold CV accuracy: 0.852
  Test accuracy:           0.847
  Test ROC-AUC:            0.652
  Best hyperparameters:    {'classifier__max_depth': None, 'classifier__n_estimators': 100}

XGBoost
  Best 5-fold CV accuracy: 0.852
  Test accuracy:           0.849
  Test ROC-AUC:            0.667
  Best hyperparameters:    {'classifier__learning_rate': 0.03, 'classifier__max_depth': 5, 'classifier__n_estimators': 100}


In [9]:
comparison = pd.DataFrame({
    name: {
        "CV accuracy": result["cv_score"],
        "Test accuracy": result["test_accuracy"],
        "Test ROC-AUC": result["test_roc_auc"],
        "Best hyperparameters": result["best_params"],
    }
    for name, result in results.items()
}).T.sort_values("Test accuracy", ascending=False)

print("\nModel comparison (ranked by held-out test accuracy):")
display(comparison)

best_model_name = comparison.index[0]
best_model = results[best_model_name]["search"].best_estimator_
print(f"Selected best_model: {best_model_name}")
print("\nBest-model test report:\n")
print(classification_report(y_test, best_model.predict(X_test), target_names=["No CHD", "CHD"], zero_division=0))



Model comparison (ranked by held-out test accuracy):


,CV accuracy,Test accuracy,Test ROC-AUC,Best hyperparameters
XGBoost,0.851709,0.849057,0.666818,"{'classifier__learning_rate': 0.03, 'classifie..."
Random Forest,0.852006,0.846698,0.651562,"{'classifier__max_depth': None, 'classifier__n..."
Logistic Regression,0.855837,0.84316,0.69389,{'classifier__C': 1.0}


Selected best_model: XGBoost

Best-model test report:

              precision    recall  f1-score   support

      No CHD       0.85      0.99      0.92       719
         CHD       0.54      0.05      0.10       129

    accuracy                           0.85       848
   macro avg       0.70      0.52      0.51       848
weighted avg       0.81      0.85      0.79       848



In [10]:
# best_model is already fitted by GridSearchCV and ready for direct predictions.
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, MODEL_PATH)
# Keep the tuned Random Forest available for deployment environments without XGBoost.
joblib.dump(results["Random Forest"]["search"].best_estimator_, MODEL_PATH.with_name("random_forest_fallback.joblib"))

metadata = {
    "model_name": best_model_name,
    "feature_names": X.columns.tolist(),
    "cv_accuracy": float(results[best_model_name]["cv_score"]),
    "test_accuracy": float(results[best_model_name]["test_accuracy"]),
    "test_roc_auc": float(results[best_model_name]["test_roc_auc"]),
    "best_hyperparameters": results[best_model_name]["best_params"],
}
METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
fallback_metadata = {
    "model_name": "Random Forest (fallback)",
    "feature_names": X.columns.tolist(),
    "test_accuracy": float(results["Random Forest"]["test_accuracy"]),
    "test_roc_auc": float(results["Random Forest"]["test_roc_auc"]),
    "best_hyperparameters": results["Random Forest"]["best_params"],
}
MODEL_PATH.with_name("random_forest_fallback_metadata.json").write_text(
    json.dumps(fallback_metadata, indent=2), encoding="utf-8"
)

print(f"Saved selected model to: {MODEL_PATH}")
print(f"Saved metadata to: {METADATA_PATH}")
# Example reuse without retraining: best_model.predict(new_patient_dataframe)

# For deployment without package-version conflicts, also export the tuned Logistic Regression as portable JSON.
portable_pipeline = results["Logistic Regression"]["search"].best_estimator_
portable_model = {
    "format": "portable_logistic_regression_v1",
    "model_name": "Portable Logistic Regression",
    "feature_names": X.columns.tolist(),
    "imputer_statistics": portable_pipeline.named_steps["imputer"].statistics_.tolist(),
    "scaler_mean": portable_pipeline.named_steps["scaler"].mean_.tolist(),
    "scaler_scale": portable_pipeline.named_steps["scaler"].scale_.tolist(),
    "coefficients": portable_pipeline.named_steps["classifier"].coef_[0].tolist(),
    "intercept": float(portable_pipeline.named_steps["classifier"].intercept_[0]),
}
MODEL_PATH.with_name("cvd_model_portable.json").write_text(json.dumps(portable_model, indent=2), encoding="utf-8")


Saved selected model to: C:\Users\rohan\Documents\Early_cvd_dectector\models\best_model.joblib
Saved metadata to: C:\Users\rohan\Documents\Early_cvd_dectector\models\best_model_metadata.json


2271